# PMM Simple Backtest
Backtest chiến lược Pure Market Making đơn giản trên Binance data

**Metrics đầu ra:**
- Net PnL (USDT)
- Sharpe Ratio
- Max Drawdown
- Total Volume
- Win Rate
- Profit Factor

## 0. Cài dependencies

In [1]:
# Chạy cell này 1 lần để cài thư viện
import subprocess
subprocess.run(['pip', 'install', 'pandas', 'numpy', 'requests', 'matplotlib', 'plotly'], check=True)
print('Done!')

PermissionError: [WinError 5] Access is denied

## 1. Import & Config

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from decimal import Decimal
from datetime import datetime

# ============================================================
# CONFIG - Thay đổi các tham số tại đây
# ============================================================
SYMBOL        = 'BTCUSDT'      # Cặp giao dịch
INTERVAL      = '1m'           # Timeframe: 1m, 5m, 15m, 1h
START_DATE    = '2025-01-01'   # Ngày bắt đầu backtest
END_DATE      = '2025-03-01'   # Ngày kết thúc backtest

# Strategy params
BID_SPREAD    = 0.001          # 0.1% bid spread
ASK_SPREAD    = 0.001          # 0.1% ask spread
ORDER_AMOUNT  = 0.01           # BTC per order
REFRESH_TIME  = 30             # giây giữa 2 lần refresh
STOP_LOSS     = 0.02           # 2% stop loss per position
TAKE_PROFIT   = 0.01           # 1% take profit per position
TRADE_COST    = 0.0006         # 0.06% maker fee Binance

# Capital
INITIAL_USDT  = 10000          # Vốn ban đầu USDT
# ============================================================

print('Config loaded OK')
print(f'  Symbol     : {SYMBOL}')
print(f'  Period     : {START_DATE} → {END_DATE}')
print(f'  Spread     : bid={BID_SPREAD*100:.2f}% ask={ASK_SPREAD*100:.2f}%')
print(f'  Order size : {ORDER_AMOUNT} BTC')
print(f'  Capital    : ${INITIAL_USDT:,} USDT')

## 2. Download Historical Data từ Binance

In [ ]:
def get_binance_klines(symbol, interval, start_date, end_date):
    """Download OHLCV data từ Binance public API"""
    url = 'https://api.binance.com/api/v3/klines'
    start_ts = int(datetime.strptime(start_date, '%Y-%m-%d').timestamp() * 1000)
    end_ts   = int(datetime.strptime(end_date,   '%Y-%m-%d').timestamp() * 1000)
    
    all_klines = []
    current_ts = start_ts
    
    print(f'Downloading {symbol} {interval} data...', end='')
    while current_ts < end_ts:
        params = {
            'symbol': symbol,
            'interval': interval,
            'startTime': current_ts,
            'endTime': end_ts,
            'limit': 1000
        }
        r = requests.get(url, params=params, timeout=10)
        data = r.json()
        if not data:
            break
        all_klines.extend(data)
        current_ts = data[-1][0] + 1
        print('.', end='', flush=True)
    
    print(f' Done! ({len(all_klines)} candles)')
    
    cols = ['open_time','open','high','low','close','volume',
            'close_time','quote_vol','trades','taker_buy_base','taker_buy_quote','ignore']
    df = pd.DataFrame(all_klines, columns=cols)
    df['open_time'] = pd.to_datetime(df['open_time'], unit='ms')
    for c in ['open','high','low','close','volume']:
        df[c] = df[c].astype(float)
    df.set_index('open_time', inplace=True)
    return df

df = get_binance_klines(SYMBOL, INTERVAL, START_DATE, END_DATE)
print(f'\nData shape: {df.shape}')
df.head(3)

## 3. Backtest Engine

Simulate logic của `pure_market_making.pyx`:
- Đặt bid/ask xung quanh mid price
- Fill nếu `low <= bid` hoặc `high >= ask`
- Stop loss / Take profit theo triple barrier
- Tính phí maker mỗi lần fill

In [ ]:
def run_backtest(df, bid_spread, ask_spread, order_amount, refresh_time_bars,
                 stop_loss_pct, take_profit_pct, trade_cost, initial_usdt):
    """
    Simulate PMM Simple strategy.
    
    Logic:
    - Mỗi refresh_time_bars: đặt lại bid/ask từ mid price
    - Bid fill khi low <= bid_price
    - Ask fill khi high >= ask_price
    - Mỗi position có stop_loss và take_profit riêng
    """
    usdt       = initial_usdt
    inventory  = 0.0      # BTC đang giữ
    trades     = []       # log từng trade
    equity     = []       # equity curve
    
    # Positions đang mở: list of dict {type, entry_price, size, stop, take_profit}
    open_positions = []
    
    bars_since_refresh = 0
    bid_active = None
    ask_active = None
    
    for i, (ts, row) in enumerate(df.iterrows()):
        mid   = row['close']
        high  = row['high']
        low   = row['low']
        
        # --- Kiểm tra stop loss / take profit cho positions đang mở ---
        closed = []
        for pos in open_positions:
            pnl = 0
            close_type = None
            if pos['type'] == 'long':  # đã mua, chờ sell
                sl_price = pos['entry'] * (1 - stop_loss_pct)
                tp_price = pos['entry'] * (1 + take_profit_pct)
                if low <= sl_price:  # stop loss hit
                    exit_price = sl_price
                    close_type = 'SL'
                elif high >= tp_price:  # take profit hit
                    exit_price = tp_price
                    close_type = 'TP'
            else:  # short
                sl_price = pos['entry'] * (1 + stop_loss_pct)
                tp_price = pos['entry'] * (1 - take_profit_pct)
                if high >= sl_price:
                    exit_price = sl_price
                    close_type = 'SL'
                elif low <= tp_price:
                    exit_price = tp_price
                    close_type = 'TP'
            
            if close_type:
                if pos['type'] == 'long':
                    gross = pos['size'] * exit_price
                    fee   = gross * trade_cost
                    usdt += gross - fee
                    inventory -= pos['size']
                    pnl = (exit_price - pos['entry']) * pos['size'] - fee
                else:
                    gross = pos['size'] * exit_price
                    fee   = gross * trade_cost
                    usdt -= gross + fee
                    inventory += pos['size']
                    pnl = (pos['entry'] - exit_price) * pos['size'] - fee
                
                trades.append({
                    'ts': ts, 'type': pos['type'], 'entry': pos['entry'],
                    'exit': exit_price, 'size': pos['size'],
                    'pnl': pnl, 'close_type': close_type
                })
                closed.append(pos)
        
        open_positions = [p for p in open_positions if p not in closed]
        
        # --- Refresh orders mỗi refresh_time_bars ---
        if bars_since_refresh >= refresh_time_bars:
            bid_active = mid * (1 - bid_spread)
            ask_active = mid * (1 + ask_spread)
            bars_since_refresh = 0
        
        # --- Check bid fill ---
        if bid_active and low <= bid_active:
            cost = bid_active * order_amount
            fee  = cost * trade_cost
            if usdt >= cost + fee:
                usdt -= (cost + fee)
                inventory += order_amount
                open_positions.append({
                    'type': 'long', 'entry': bid_active, 'size': order_amount
                })
                bid_active = None  # reset sau khi fill
        
        # --- Check ask fill ---
        if ask_active and high >= ask_active:
            if inventory >= order_amount:
                revenue = ask_active * order_amount
                fee     = revenue * trade_cost
                usdt += (revenue - fee)
                inventory -= order_amount
                open_positions.append({
                    'type': 'short', 'entry': ask_active, 'size': order_amount
                })
                ask_active = None
        
        # --- Equity = USDT + mark-to-market inventory ---
        equity.append(usdt + inventory * mid)
        bars_since_refresh += 1
    
    return pd.DataFrame(trades), pd.Series(equity, index=df.index)


# Convert refresh_time (seconds) → số bars
bar_seconds = {'1m': 60, '5m': 300, '15m': 900, '1h': 3600}
refresh_bars = max(1, REFRESH_TIME // bar_seconds.get(INTERVAL, 60))

print(f'Running backtest ({len(df)} bars, refresh every {refresh_bars} bars)...')
trades_df, equity_curve = run_backtest(
    df, BID_SPREAD, ASK_SPREAD, ORDER_AMOUNT,
    refresh_bars, STOP_LOSS, TAKE_PROFIT,
    TRADE_COST, INITIAL_USDT
)
print(f'Done! Total trades: {len(trades_df)}')

## 4. Performance Metrics

In [ ]:
def calc_metrics(trades_df, equity_curve, initial_capital):
    if len(trades_df) == 0:
        print('No trades!')
        return {}
    
    # PnL
    net_pnl      = trades_df['pnl'].sum()
    net_pnl_pct  = net_pnl / initial_capital * 100
    total_volume = (trades_df['entry'] * trades_df['size']).sum() * 2  # buy + sell
    
    # Win/Loss
    wins         = trades_df[trades_df['pnl'] > 0]
    losses       = trades_df[trades_df['pnl'] < 0]
    win_rate     = len(wins) / len(trades_df) * 100
    profit_factor = wins['pnl'].sum() / abs(losses['pnl'].sum()) if len(losses) > 0 else float('inf')
    
    # Drawdown
    rolling_max  = equity_curve.cummax()
    drawdown     = (equity_curve - rolling_max) / rolling_max * 100
    max_dd       = drawdown.min()
    
    # Sharpe (annualized, dùng bar returns)
    returns      = equity_curve.pct_change().dropna()
    bars_per_year = {'1m': 525600, '5m': 105120, '15m': 35040, '1h': 8760}
    ann_factor   = bars_per_year.get(INTERVAL, 525600)
    sharpe       = (returns.mean() / returns.std()) * np.sqrt(ann_factor) if returns.std() > 0 else 0
    
    # Close types
    close_types  = trades_df['close_type'].value_counts().to_dict()
    
    metrics = {
        'Net PnL (USDT)':    round(net_pnl, 2),
        'Net PnL (%)':       round(net_pnl_pct, 2),
        'Total Volume (USDT)': round(total_volume, 0),
        'Total Trades':      len(trades_df),
        'Win Rate (%)':      round(win_rate, 1),
        'Profit Factor':     round(profit_factor, 2),
        'Sharpe Ratio':      round(sharpe, 2),
        'Max Drawdown (%)':  round(max_dd, 2),
        'Close Types':       close_types,
    }
    return metrics

metrics = calc_metrics(trades_df, equity_curve, INITIAL_USDT)

print('=' * 45)
print(f'  BACKTEST RESULTS: {SYMBOL} {START_DATE}→{END_DATE}')
print('=' * 45)
for k, v in metrics.items():
    print(f'  {k:<25}: {v}')
print('=' * 45)

## 5. Visualize

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
fig.suptitle(f'PMM Simple Backtest — {SYMBOL} {START_DATE} → {END_DATE}', fontsize=13)

# --- 1. Equity Curve ---
axes[0].plot(equity_curve.index, equity_curve.values, color='steelblue', linewidth=1)
axes[0].axhline(INITIAL_USDT, color='gray', linestyle='--', linewidth=0.8)
axes[0].set_title('Equity Curve (USDT)')
axes[0].set_ylabel('USDT')
axes[0].grid(alpha=0.3)

# --- 2. Drawdown ---
rolling_max = equity_curve.cummax()
drawdown    = (equity_curve - rolling_max) / rolling_max * 100
axes[1].fill_between(drawdown.index, drawdown.values, 0, color='tomato', alpha=0.5)
axes[1].set_title('Drawdown (%)')
axes[1].set_ylabel('%')
axes[1].grid(alpha=0.3)

# --- 3. Cumulative PnL per trade ---
if len(trades_df) > 0:
    cum_pnl = trades_df['pnl'].cumsum()
    colors  = ['green' if x > 0 else 'red' for x in trades_df['pnl']]
    axes[2].bar(range(len(cum_pnl)), trades_df['pnl'].values, color=colors, alpha=0.6)
    axes[2].plot(range(len(cum_pnl)), cum_pnl.values, color='navy', linewidth=1.5)
    axes[2].axhline(0, color='gray', linestyle='--', linewidth=0.8)
    axes[2].set_title('PnL per Trade & Cumulative PnL')
    axes[2].set_ylabel('USDT')
    axes[2].set_xlabel('Trade #')
    axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('backtest_result.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart saved: backtest_result.png')

## 6. Optimize Spread với Grid Search

In [ ]:
# Grid search qua các mức spread khác nhau
spread_range = [0.0005, 0.001, 0.0015, 0.002, 0.003, 0.005]
results = []

print('Running grid search...')
for bid in spread_range:
    for ask in spread_range:
        t, eq = run_backtest(df, bid, ask, ORDER_AMOUNT, refresh_bars,
                             STOP_LOSS, TAKE_PROFIT, TRADE_COST, INITIAL_USDT)
        if len(t) > 0:
            ret   = eq.pct_change().dropna()
            sharpe = (ret.mean() / ret.std()) * np.sqrt(525600) if ret.std() > 0 else 0
            dd     = ((eq - eq.cummax()) / eq.cummax()).min() * 100
            results.append({
                'bid_spread': bid,
                'ask_spread': ask,
                'net_pnl':    round(t['pnl'].sum(), 2),
                'trades':     len(t),
                'sharpe':     round(sharpe, 2),
                'max_dd_pct': round(dd, 2),
            })

results_df = pd.DataFrame(results).sort_values('sharpe', ascending=False)
print('\nTop 10 combinations by Sharpe:')
print(results_df.head(10).to_string(index=False))